# 03 · Dinamik değişken seçimi

Alan listesi koda gömülü değildir. JSON sözlüğü kaynak sütunu, türü, rolü, zamanında bilinirliği
ve toplulaştırmayı tanımlar. CatBoost destekli geçmiş dönem kontrolleri, eğitimde kalite /
korelasyon elemesi ve tekrarlı permütasyon katkılarıyla ortak bir kümeleme aday seti çıkarılır.
Geçmiş stok tahmin önemleriyle bu yapısal seçim aynı deney değildir.


In [ ]:
from pathlib import Path
import sys
_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "python_kodlari/ayarlar").is_dir() and (p / "notebooks/jupyter_araclari.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Projenin tamamını indirin; notebook’u projenin notebooks klasöründen açın.")
for _folder in [ROOT / "python_kodlari", ROOT / "notebooks"]:
    if str(_folder) not in sys.path:
        sys.path.insert(0, str(_folder))
from IPython.display import display, Markdown
from jupyter_araclari import (read_json, read_table, latest_run, fresh_run, record_run,
    result_path, require_files, prepare_configs, source_input, run_python, extract_bundle,
    snapshot, current_decision, qualified_selection)
print("Proje:", ROOT)
print("Kernel Python:", sys.executable)


In [ ]:
import pandas as pd
SECIMI_CALISTIR = False
VERI_DOSYASI = ROOT / "outputs/banka_test_verisi_v4/sube_model_girdileri_aylik.csv"
config = read_json(ROOT / "python_kodlari/ayarlar/dinamik_v4.json")
# Gerçek veri için config["input"]["keys"], config["fields"] ve zaman kurallarını düzenleyin.
display(pd.DataFrame(config["fields"])[["name", "source", "dtype", "role"]])
display(config["selection"])


## Alan sözlüğü taslağı (kendi veri dosyan için isteğe bağlı)

Taslak anlamsal rolü senin yerine onaylamaz. Müşteri / şube düzeyini, anahtarları, hedefleri
ve aday alanları kontrol edip sözlüğü tamamla. Bu hücre mevcut ayar dosyasının üzerine yazmaz.


In [ ]:
SOZLUK_TASLAGI_URET = False
YENI_VERI = VERI_DOSYASI
if SOZLUK_TASLAGI_URET:
    require_files(YENI_VERI)
    folder = fresh_run("sozluktaslagi")
    folder.mkdir()
    run_python("dinamik_feature_selection.py", "discover", "--input", YENI_VERI, "--output", folder / "alan_sozlugu.json", "--branch", "sube_id", "--period", "donem", "--group", "buyuk_sehir_grubu")
    print("Düzenlenecek sözlük:", folder / "alan_sozlugu.json")


## Seçimi çalıştır

Orijinal JSON dosyaları değiştirilmez. Bağlantılı ayarlar birlikte kopyalanır ve veri yolları
mutlaklaştırılır; notebook başka klasörden açılsa da aynı dosya kullanılır. `run` kendi
modellerini, permütasyon sonuçlarını ve kayıtlı tahminleri bağımsız doğrular.


In [ ]:
if SECIMI_CALISTIR:
    require_files(VERI_DOSYASI)
    from dinamik_secim.runner import run as select_features
    configs = prepare_configs(VERI_DOSYASI, {"dinamik_v4.json": config})
    selection = select_features(configs / "dinamik_v4.json", fresh_run("dinamik_feature_selection"))
    print("Tamamlanan seçim:", selection)
else:
    print("Eğitim yapılmadı; hazır seçim tabloları gösteriliyor.")


In [ ]:
schema = read_json(result_path("dinamik_feature_selection", "selected_schema.json", "selection_schema.json"))
decisions = read_table(result_path("dinamik_feature_selection", "decisions.csv", "selection_decisions.csv"))
print("Seçilmiş alan sayısı:", len(schema["features"]))
display(decisions[["name", "family", "dtype", "status", "reason"]])
display(pd.DataFrame({"sıra": range(1, len(schema["features"]) + 1), "alan": schema["features"]}))


In [ ]:
importance = read_table(result_path("dinamik_feature_selection", "importance_repeats.csv", "selection_importance_repeats.csv"))
summary = importance.groupby(["feature", "target"]).delta_wape.mean().unstack()
display(summary.sort_index())
print("Pozitif delta WAPE: alan bozulduğunda doğrulama hatası artmış. Tek başına nedensellik kanıtı değildir.")
display(read_json(result_path("dinamik_feature_selection", "drift.json", "selection_drift.json")))


Seçilen şema sonraki adımın girdisidir. Her ay aynı donmuş şema uygulanabilir; yeni alan
eklemek veya şemayı değiştirmek yeniden seçim gerektirir. Bu notebook küme veya hedef üretmez.
Sonraki adım **04**.
